# Transactions - Bronze
- Requester Note: There might schema drift and Schema Evolution (New Columns may be added)
- Partitioned it by Transaction Month
- Flatten Nested Columns

**Reads:** `bronze.bronze_transactions` (rows for `BATCH_ID`)  
**Writes:** `validate.bronze_clean_transactions` (append, partitioned by `txn_month`)  
**Parameter:** `BATCH_ID`

## Imports

In [ ]:
# Imports
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructField, StructType

## Shared functions

In [ ]:
%run NB_FUNCTIONS

## Parameters

The next cell is tagged `parameters`, so a pipeline can override it at run time.

In [ ]:
BATCH_ID = ""

## Transaction Dataframe

In [ ]:
txn_df = spark.read.table("bronze.bronze_transactions").filter(
    F.col("_batch_id") == BATCH_ID
)

In [ ]:
display(txn_df)

## Handling Schema Drift

In [ ]:
COLUMN_MAPPING = {
    "txn_amount": "amount",
    "transaction_timestamp": "txn_ts",
    "transaction_id": "txn_id",
    "transaction_type": "txn_type",
}

for old_name, new_name in COLUMN_MAPPING.items():
    if old_name in txn_df.columns and new_name not in txn_df.columns:
        txn_df = txn_df.withColumnRenamed(old_name, new_name)

## Normalise Null Tokens

In [ ]:
txn_n = normalise_null(txn_df)

## Standardizing Category Attributes

In [ ]:
TXN_TYPE = {
    "DEBIT": "DEBIT",
    "DR": "DEBIT",
    "D": "DEBIT",
    "CREDIT": "CREDIT",
    "CR": "CREDIT",
    "C": "CREDIT",
    "REFUND": "REFUND",
    "RFND": "REFUND",
}

CURRENCY = {
    "INR": "INR",
    "RS": "INR",
    "₹": "INR",
    "USD": "USD",
    "$": "USD",
    "US$": "USD",
    "EUR": "EUR",
    "GBP": "GBP",
    "AED": "AED",
    "SGD": "SGD",
}

CHANNEL = {c: c for c in ["POS", "ECOM", "ATM", "UPI", "NEFT", "IMPS"]}

txn_c = (
    txn_n.withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("raw_amount", F.col("amount"))
    .withColumn("amount", amount_parse("amount"))
    .withColumn("currency", mapping_values("currency", CURRENCY, "UNKNOWN"))
    .withColumn("txn_type", mapping_values("txn_type", TXN_TYPE, "UNKNOWN"))
    .withColumn("channel", mapping_values("channel", CHANNEL, "UNKNOWN"))
    .withColumn("status", F.upper(F.trim("status")))
)

## Parsing Transaction Timestamp

In [ ]:
txn_t = (
    txn_c.withColumn("raw_txn_ts", F.col("txn_ts"))
    .withColumn("txn_ts", parse_txn_ts("txn_ts"))
    .withColumn("updated_at", parse_txn_ts("updated_at"))
    .withColumn(
        "txn_date_ist", F.to_date(F.from_utc_timestamp("txn_ts", "Asia/Kolkata"))
    )
)

## Flatten Nested Values

In [ ]:
txn_f = flatten(txn_t)

## Explode tags

In [ ]:
txn_enriched = (
    txn_f.withColumn(
        "is_international",
        F.array_contains("tags", "international") | (F.col("currency") != "INR"),
    )
    .withColumn("tag_count", F.size(F.coalesce("tags", F.array())))
    .withColumn("tags_text", F.array_join("tags", ","))
)

## Cleaning Merchants attributes

In [ ]:
txn_enriched = (
    txn_enriched.withColumn("merchant_name_raw", F.col("merchant_name"))
    .withColumn(
        "merchant_name",
        F.initcap(F.regexp_replace(F.trim(F.col("merchant_name")), r"\s+", " ")),
    )
    .withColumn(
        "mcc", F.when(F.col("merchant_mcc").rlike(r"^\d{4}$"), F.col("merchant_mcc"))
    )
    .withColumn(
        "mcc_invalid", F.col("mcc").isNull() & F.col("merchant_mcc").isNotNull()
    )
)

## Drop Duplicates Transactions

In [ ]:
drop_cols = [cols for cols in txn_enriched.columns if not cols.startswith("_")]
before = txn_enriched.count()
txn_d = txn_enriched.dropDuplicates(drop_cols)
after = txn_d.count()
print(before, after)

## Keeping Latest update as per business key

In [ ]:
txn_l = latest_by_key(txn_d, ["txn_id"], ["updated_at"])

## Partitioning by Month and Processing to Validate Staging Layer

In [ ]:
txn_cleaned = txn_l.withColumn("txn_month", F.date_format("txn_date_ist", "yyyy-MM"))

spark.sql("CREATE SCHEMA IF NOT EXISTS validate")
txn_cleaned.write.mode("append").partitionBy("txn_month").saveAsTable(
    "validate.bronze_clean_transactions"
)